## 0. Preparación

In [ ]:
import json
import random
import re
import time
from datetime import date, datetime, timezone
from enum import Enum
from pathlib import Path
from typing import Any

import pandas as pd
from pydantic import BaseModel, Field
from pydantic_ai import Agent

from renewables_permitting.utils import (
    normalize_text,
    save_parquet,
    validate_required_columns,
)

BASE_URL = "https://www.boe.es/datosabiertos/api/boe/sumario"

# PROJECT_ROOT = Path(__file__).resolve().parents[2]  # fuera del notebook
PROJECT_ROOT = Path.cwd().parent  # dentro del notebook

DATA_DIR = PROJECT_ROOT / "data"

BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

# BRONZE
BOE_DOCS_XML_DIR = BRONZE_DIR / "boe_docs_xml"


# SILVER
BOE_CANDIDATES_PATH = SILVER_DIR / "boe_candidates" / "boe_candidates_normalized.parquet"

BOE_CANDIDATES_DOCS_TEXT_PATH = SILVER_DIR / "boe_candidates_docs_text" / "boe_candidates_docs_text.parquet"

DIM_MUNICIPALITIES_PATH = SILVER_DIR / "dimensions" / "dim_municipalities.parquet"

SILVER_BOE_AI_DIR = SILVER_DIR / "boe_ai"

BOE_AI_EXTRACTIONS_PATH = SILVER_BOE_AI_DIR / "boe_ai_extractions.parquet"
LIFECYCLE_EVENTS_PATH = SILVER_BOE_AI_DIR / "lifecycle_events.parquet"
ADMINISTRATIVE_ACTIONS_PATH = SILVER_BOE_AI_DIR / "administrative_actions.parquet"
ASSET_MENTIONS_PATH = SILVER_BOE_AI_DIR / "asset_mentions.parquet"
ASSET_TECHNOLOGIES_PATH = SILVER_BOE_AI_DIR / "asset_technologies.parquet"
ASSET_PARTICIPANTS_PATH = SILVER_BOE_AI_DIR / "asset_participants.parquet"
ASSET_LOCATIONS_PATH = SILVER_BOE_AI_DIR / "asset_locations.parquet"
ASSET_ALIASES_PATH = SILVER_BOE_AI_DIR / "asset_aliases.parquet"
ASSET_RELATION_MENTIONS_PATH = SILVER_BOE_AI_DIR / "asset_relation_mentions.parquet"


# GOLD
PROJECT_GROUPS_PATH = GOLD_DIR / "project_groups.parquet"
PROJECT_ASSETS_PATH = GOLD_DIR / "project_assets.parquet"
PROJECT_TIMELINE_PATH = GOLD_DIR / "project_timeline.parquet"
PROJECT_STATUS_PATH = GOLD_DIR / "project_status.parquet"

## 1. Instrucciones

In [ ]:
INSTRUCTIONS = """
Eres un extractor de información estructurada de documentos del BOE sobre proyectos energéticos.
Devuelve exclusivamente JSON válido conforme al esquema BOEProjectExtraction.

Objetivo:
Extraer eventos de ciclo de vida de proyectos energéticos a partir de una publicación del BOE.

Reglas generales:
- Extrae únicamente información explícitamente contenida en el título o en el texto del documento.
- No inventes, completes ni corrijas datos por conocimiento externo.
- Si un dato no aparece, usa null, lista vacía o no_consta, según corresponda al esquema.
- Toda información relevante debe estar respaldada por evidencia textual.
- No asignes identificadores definitivos de proyecto.
- Usa únicamente identificadores locales internos al documento para relacionar activos dentro de la misma publicación: asset_1, asset_2, asset_3, ...
- No asumas que activos con nombres similares pertenecen al mismo proyecto si el documento no lo indica explícitamente.
- No intentes resolver si varias publicaciones pertenecen al mismo proyecto global; limita la extracción a la información explícita contenida en la publicación actual.
- Distingue entre hechos publicados en el documento y antecedentes históricos mencionados únicamente como contexto.
- Conserva las relaciones explícitas entre activos, promotores, localizaciones y procedimientos cuando aparezcan en el texto.
- Si existe ambigüedad, conserva la información observada y no fuerces una interpretación.
- event_summary debe contener una frase breve y no nula que describa la evolución principal del proyecto o activo energético afectado.

Interpretación numérica:
- Interpreta los números con formato español.
- "31,172 MW" equivale a 31.172 MW.
- "28.000 kW" equivale a 28000 kW.
- Cuando el texto incluya potencia unitaria y número de equipos, calcula la potencia total normalizada si la equivalencia es explícita.
- Si existe discrepancia aparente entre el cálculo explícito y una cifra textual, no afirmes que el BOE contiene una errata.
- Conserva la discrepancia en power_normalization_note.
- Ejemplo: "14 aerogeneradores de 2000 kW" se normaliza como 28 MW; si el texto además dice "28.000 MW", indícalo en la nota sin corregir ni calificar el texto.

Relevancia:
- Clasifica como relevante solo documentos vinculados a proyectos energéticos concretos.
- Un documento es relevante si trata sobre generación eléctrica renovable, almacenamiento, infraestructuras de evacuación, líneas eléctricas, subestaciones, autorizaciones administrativas, evaluación ambiental, declaración de impacto ambiental, informe de determinación de afección ambiental, declaración de utilidad pública, expropiación, archivo, denegación, modificación, ampliación o repotenciación de un proyecto concreto.
- Clasifica como no_relevante documentos normativos, estadísticos, tarifarios, presupuestarios, genéricos o no vinculados a un proyecto energético identificable.
- Usa dudoso cuando exista vocabulario energético pero no haya información suficiente para identificar un proyecto tramitado.

Unidad de extracción:
- La unidad principal es lifecycle_events.
- Cada publicación BOE debe generar normalmente un único lifecycle_event principal.
- Un lifecycle_event representa una evolución material o funcional del proyecto o activo energético y los actos administrativos publicados sobre ella.
- Solo crea varios lifecycle_events cuando la publicación describa varias evoluciones materiales independientes o afecte a activos principales distintos.
- No crees un lifecycle_event separado para un trámite administrativo si ya está recogido en administrative_actions.
- Si una publicación contiene varios actos administrativos sobre la misma evolución del proyecto, inclúyelos en administrative_actions dentro del mismo lifecycle_event.
- Ejemplo: si una publicación otorga AAP y AAC para el mismo parque, crea un único lifecycle_event con dos administrative_actions.
- Ejemplo: si el documento formula un informe de determinación de afección ambiental para una hibridación, usa event_type = hybridization y añade una administrative_action con procedure_stage = informe_determinacion_afeccion_ambiental y procedure_decision = formulado.
- No añadas otro lifecycle_event con event_type = other o unknown para el mismo hecho.

Procedimiento administrativo:
- administrative_actions debe recoger los actos administrativos publicados en el documento.
- Cada administrative_action debe tener procedure_stage, procedure_decision y evidence.
- procedure_stage debe reflejar el trámite publicado en el BOE.
- procedure_decision debe reflejar la decisión principal: formulado, favorable, desfavorable, autorizado, denegado, sometido_informacion_publica, declarado_utilidad_publica, archivado, desistido, inadmitido, etc.
- No incluyas antecedentes menores salvo que sean necesarios para interpretar el evento.
- No dupliques como lifecycle_event lo que ya esté expresado como procedure_stage o procedure_decision.

Activos:
- Extrae solo activos relevantes para entender la evolución del proyecto.
- Extrae nuevas instalaciones de generación.
- Extrae instalaciones existentes afectadas por hibridación, modificación, ampliación o repotenciación.
- Extrae sistemas de almacenamiento.
- Extrae infraestructuras energéticas solo si son el objeto principal del documento.
- No extraigas líneas, subestaciones, centros de seccionamiento ni puntos de conexión como assets si solo aparecen como infraestructura auxiliar de evacuación, conexión, modificación técnica o acceso a red.
- Tampoco crees asset_relations hacia infraestructuras auxiliares.
- Si el documento trata principalmente de una línea, subestación, infraestructura de evacuación o expropiación asociada a esa infraestructura, sí puedes extraerla como asset principal.
- Cada asset debe incluir evidence textual breve que justifique su existencia, nombre, potencia o papel en el evento.
- No dejes evidence = null en un asset salvo que el activo sea dudoso o la evidencia no pueda aislarse con claridad.
- La evidence del asset debe ser específica del activo, no una cita genérica de toda la resolución.

Relaciones entre activos:
- Usa asset_relations únicamente entre activos energéticos principales o activos existentes afectados por la evolución del proyecto.
- No uses asset_relations para describir toda la red de evacuación, conexión o acceso.
- Usa hybridizes_with cuando una instalación nueva hibrida con una instalación existente.
- Usa adds_technology_to cuando se añade una nueva tecnología a un activo o complejo existente.
- Usa modifies cuando se modifica un activo existente.
- Usa expands cuando se amplía un activo existente.
- Usa repowers cuando se repotencia un activo existente.
- Usa adds_storage_to cuando se añade almacenamiento a un activo existente.
- Usa same_project_group_as cuando el texto indica explícitamente que varios activos forman parte del mismo proyecto o complejo.
- Usa associated_with solo para relaciones explícitas relevantes que no encajen en las anteriores.

Hibridación:
- Si el documento describe una hibridación, crea un asset para la nueva instalación.
- Crea otro asset para el activo existente afectado si aparece en el texto.
- Marca la nueva instalación como new_asset.
- Marca el activo previo como existing_asset o in_operation si el texto indica que está en operación.
- Crea una relación hybridizes_with desde la nueva instalación hacia el activo existente.
- Si el texto indica que la nueva tecnología se añade al complejo o activo existente, añade también adds_technology_to.
- No crees assets ni relaciones para subestaciones, líneas o puntos de conexión auxiliares de la hibridación.

Promotores y participantes:
- Extrae promotores, copromotores, titulares u operadores solo si aparecen explícitamente.
- Puede haber varios participantes.
- No asumas que el promotor de un activo es también promotor de otro si el texto no lo dice.
- Conserva la denominación literal de la entidad.
- No inventes CIF, NIF ni identificadores societarios.
- No resuelvas entidades empresariales por conocimiento externo.

Municipios:
- Para cada municipio identificado, extrae el nombre mencionado en el documento.
- Extrae provincia y comunidad autónoma solo si aparecen en el documento.
- Llama siempre a la herramienta resolve_municipality.
- Utiliza exclusivamente el resultado devuelto por resolve_municipality para completar municipio oficial, provincia, comunidad autónoma y códigos INE.
- No inventes códigos INE ni divisiones administrativas.
- No incluyas municipios que no aparezcan explícitamente como ubicación del activo.
- No incluyas municipios mencionados solo en direcciones postales, sedes sociales o antecedentes no relacionados con la ubicación del activo.
"""

## 2. Contratos de salida

Qué quiero saber de cada publicación?

### Clasificación documental

In [4]:
# Clasificación global de relevancia energética.

class RelevanciaEnergetica(str, Enum):
    RELEVANTE = "relevante"
    NO_RELEVANTE = "no_relevante"
    DUDOSO = "dudoso"
    
# relevante:
#   Documento sobre generación eléctrica renovable, almacenamiento, evacuación, subestaciones, líneas eléctricas o autorizaciones ambientales/administrativas asociadas.

# no_relevante:
#   Documento energético genérico, normativo, estadístico, tarifario, presupuestario o no vinculado a un proyecto concreto.

# dudoso:
#   Documento con vocabulario energético, pero sin información suficiente para saber si corresponde a un proyecto tramitado.

### Tecnologías e instalaciones energéticas

In [5]:
# Tecnologías de generación y almacenamiento.
class TechnologyType(str, Enum):
    FOTOVOLTAICA = "fotovoltaica"
    EOLICA = "eolica"
    TERMOSOLAR = "termosolar"
    HIDROELECTRICA = "hidroelectrica"
    GEOTERMICA = "geotermica"
    BIOMASA = "biomasa"
    BIOGAS = "biogas"
    HIDROGENO_VERDE = "hidrogeno_verde"
    ALMACENAMIENTO = "almacenamiento"
    OTRA = "otra"
    DESCONOCIDA = "desconocida"

In [6]:
# Características técnicas de una tecnología.
class Technology(BaseModel):
    technology_type: TechnologyType
    installed_power_mw: float | None = None
    peak_power_mwp: float | None = None
    description: str | None = None
    power_normalization_note: str | None = None

In [7]:
# Sistemas de almacenamiento energético.
class StorageSystem(BaseModel):
    exists: bool = False
    power_mw: float | None = None
    capacity_mwh: float | None = None
    description: str | None = None

In [8]:
# class InfrastructureType(str, Enum):
#     SUBESTACION = "subestacion"
#     LINEA_ELECTRICA = "linea_electrica"
#     CENTRO_SECCIONAMIENTO = "centro_seccionamiento"
#     INFRAESTRUCTURA_EVACUACION = "infraestructura_evacuacion"
#     OTRA = "otra"
#     DESCONOCIDA = "desconocida"

# class Infrastructure(BaseModel):
#     type: InfrastructureType
#     name: str | None = None
#     voltage_kv: float | None = None
#     length_km: float | None = None

In [9]:
# class HybridConfiguration(BaseModel):
#     is_hybrid: bool = False
#     generation_technology_types: list[TechnologyType] = Field(default_factory=list)
#     includes_storage: bool = False
#     description: str | None = None

### Localización administrativa (INE)

Usar Pydantic AI solo para extraer candidatos textuales y contexto; la validación final debe hacerla una función determinista.

In [10]:
# Estado de resolución administrativa.
class MunicipalityResolutionStatus(str, Enum):
    RESOLVED = "resolved"
    AMBIGUOUS = "ambiguous"
    NOT_FOUND = "not_found"


# Municipio normalizado mediante catálogo INE.
class MunicipalityLocation(BaseModel):
    ine_municipality_code: str
    municipality: str
    ine_province_code: str | None = None
    province: str | None = None
    ine_autonomous_community_code: str | None = None
    autonomous_community: str | None = None
    

# Resultado de resolución de municipios.    
class MunicipalityLookupResult(BaseModel):
    query: str
    province_hint: str | None = None
    autonomous_community_hint: str | None = None

    resolution_status: MunicipalityResolutionStatus
    resolved: MunicipalityLocation | None = None
    candidates: list[MunicipalityLocation] = Field(default_factory=list)

    matched_by: str | None = None
    reason: str | None = None

### Procedimiento administrativo

In [11]:
# Tipo de trámite administrativo.
class ProcedureStage(str, Enum):
    # Inicio y antecedentes del procedimiento
    SOLICITUD_TRAMITACION = "solicitud_tramitacion"
    SOLICITUD_TRAMITACION_AMBIENTAL = "solicitud_tramitacion_ambiental"
    SUBSANACION_DOCUMENTACION = "subsanacion_documentacion"
    VERIFICACION_REQUISITOS_TRAMITACION = "verificacion_requisitos_tramitacion"

    # Información pública
    INFORMACION_PUBLICA = "informacion_publica"

    # Evaluación ambiental
    DECLARACION_IMPACTO_AMBIENTAL = "declaracion_impacto_ambiental"
    INFORME_DETERMINACION_AFECCION_AMBIENTAL = "informe_determinacion_afeccion_ambiental"

    # Autorizaciones energéticas
    AUTORIZACION_ADMINISTRATIVA_PREVIA = "autorizacion_administrativa_previa"
    AUTORIZACION_ADMINISTRATIVA_CONSTRUCCION = "autorizacion_administrativa_construccion"
    AUTORIZACION_EXPLOTACION = "autorizacion_explotacion"

    # Utilidad pública y expropiación
    DECLARACION_UTILIDAD_PUBLICA = "declaracion_utilidad_publica"
    EXPROPIACION_FORZOSA = "expropiacion_forzosa"
    RELACION_BIENES_DERECHOS_AFECTADOS = "relacion_bienes_derechos_afectados"
    LEVANTAMIENTO_ACTAS_PREVIAS_OCUPACION = "levantamiento_actas_previas_ocupacion"
    ACTAS_OCUPACION = "actas_ocupacion"

    # Modificaciones y terminación anormal
    MODIFICACION = "modificacion"
    ARCHIVO_EXPEDIENTE = "archivo_expediente"
    DESISTIMIENTO = "desistimiento"
    INADMISION = "inadmision"

    # Fallback
    OTRO = "otro"
    NO_CONSTA = "no_consta"

In [12]:
# Resultado o decisión administrativa.
class ProcedureDecision(str, Enum):
    # Inicio y tramitación del expediente
    SOLICITADO = "solicitado"
    SUBSANADO = "subsanado"
    REQUISITOS_VERIFICADOS = "requisitos_verificados"
    MODIFICADO = "modificado"
    PRORROGADO = "prorrogado"
    FORMULADO = "formulado"

    # Información pública y participación
    SOMETIDO_INFORMACION_PUBLICA = "sometido_informacion_publica"
    CONVOCADO = "convocado"

    # Evaluación ambiental
    FAVORABLE = "favorable"
    DESFAVORABLE = "desfavorable"
    SOMETIDO_EIA_ORDINARIA = "sometido_eia_ordinaria"
    NO_SOMETIDO_EIA_ORDINARIA = "no_sometido_eia_ordinaria"
    DECLARADO_UTILIDAD_PUBLICA = "declarado_utilidad_publica"

    # Resolución administrativa
    APROBADO = "aprobado"
    AUTORIZADO = "autorizado"
    DENEGADO = "denegado"

    # Terminación anormal del procedimiento
    ARCHIVADO = "archivado"
    DESISTIDO = "desistido"
    INADMITIDO = "inadmitido"

    # Información no disponible
    NO_CONSTA = "no_consta"

In [13]:
# Acto administrativo publicado en el BOE.
class AdministrativeAction(BaseModel):
    procedure_stage: ProcedureStage = ProcedureStage.NO_CONSTA  # "el campo tiene tipo ProcedureStage y valor por defecto NO_CONSTA"
    procedure_decision: ProcedureDecision = ProcedureDecision.NO_CONSTA
    evidence: str | None = None

### Participantes

In [14]:
# Rol de una entidad participante.
class ParticipantRole(str, Enum):
    PROMOTER = "promoter"
    CO_PROMOTER = "co_promoter"
    OWNER = "owner"
    OPERATOR = "operator"
    GRID_OWNER = "grid_owner"
    ADMINISTRATION = "administration"
    UNKNOWN = "unknown"

In [15]:
# Empresa, administración o entidad participante.
class ProjectParticipant(BaseModel):
    name: str
    role: ParticipantRole = ParticipantRole.UNKNOWN
    evidence: str | None = None

### Activos energéticos

In [16]:
# Papel del activo dentro del evento.
class AssetRole(str, Enum):
    NEW_ASSET = "new_asset"
    EXISTING_ASSET = "existing_asset"
    MODIFIED_ASSET = "modified_asset"
    AFFECTED_ASSET = "affected_asset"
    ASSOCIATED_ASSET = "associated_asset"
    MAIN_ASSET = "main_asset"
    UNKNOWN = "unknown"

In [17]:
# Estado administrativo u operativo del activo.
class AssetStatus(str, Enum):
    PLANNED = "planned"
    UNDER_PERMITTING = "under_permitting"
    AUTHORIZED = "authorized"
    UNDER_CONSTRUCTION = "under_construction"
    IN_OPERATION = "in_operation"
    EXISTING = "existing"
    DENIED = "denied"
    ARCHIVED = "archived"
    UNKNOWN = "unknown"

In [18]:
# Instalación energética identificada en el documento.
class EnergyAsset(BaseModel):
    local_asset_id: str
    name: str | None = None
    aliases: list[str] = Field(default_factory=list)

    role_in_event: AssetRole = AssetRole.UNKNOWN
    status_in_document: AssetStatus = AssetStatus.UNKNOWN

    technologies: list[Technology] = Field(default_factory=list)
    storage_systems: list[StorageSystem] = Field(default_factory=list)
    participants: list[ProjectParticipant] = Field(default_factory=list)
    locations: list[MunicipalityLocation] = Field(default_factory=list)

    evidence: str | None = None


### Relaciones entre activos

In [19]:
# Tipo de relación entre activos energéticos.
class AssetRelationType(str, Enum):
    HYBRIDIZES_WITH = "hybridizes_with"
    ADDS_TECHNOLOGY_TO = "adds_technology_to"
    MODIFIES = "modifies"
    EXPANDS = "expands"
    REPOWERS = "repowers"
    ADDS_STORAGE_TO = "adds_storage_to"
    SHARES_GRID_ACCESS_WITH = "shares_grid_access_with"
    ASSOCIATED_WITH = "associated_with"
    SAME_PROJECT_GROUP_AS = "same_project_group_as"
    UNKNOWN = "unknown"

In [20]:
# Relación explícita entre activos.
class AssetRelation(BaseModel):
    source_asset_id: str
    target_asset_id: str
    relation_type: AssetRelationType
    evidence: str | None = None

### Evolución del proyecto

In [21]:
# Tipo de evolución material del proyecto.
class LifecycleEventType(str, Enum):
    NEW_PROJECT = "new_project"
    HYBRIDIZATION = "hybridization"
    MODIFICATION = "modification"
    EXPANSION = "expansion"
    REPOWERING = "repowering"
    STORAGE_ADDITION = "storage_addition"
    EVACUATION_INFRASTRUCTURE = "evacuation_infrastructure"
    OWNERSHIP_CHANGE = "ownership_change"
    OTHER = "other"
    UNKNOWN = "unknown"

In [22]:
# Evento principal de ciclo de vida del proyecto.
# Debería haber un LifecycleEvent por cada objeto administrativo (activo).
class ProjectLifecycleEvent(BaseModel):
    event_type: LifecycleEventType = LifecycleEventType.UNKNOWN
    administrative_actions: list[AdministrativeAction] = Field(default_factory=list)
    assets: list[EnergyAsset] = Field(default_factory=list)
    asset_relations: list[AssetRelation] = Field(default_factory=list)
    event_summary: str | None = None
    evidence: str | None = None

### Documento BOE extraído

In [ ]:
class BOEProjectExtraction(BaseModel):
    identificador_boe: str
    fecha_publicacion: date | None = None

    relevancia_energetica: RelevanciaEnergetica
    es_relevante_para_proyecto: bool
    relevance_reason: str | None = None

    lifecycle_events: list[ProjectLifecycleEvent] = Field(default_factory=list)

    extraction_notes: str | None = None

### Notas para la estimación objetiva de la confianza

In [24]:
# TODO: Estimar objetivamente la confianza de la extracción con IA
# confidence = 1.0

# confidence = 1.0

# if project_name is None:
#     confidence -= 0.2

# if promoter is None:
#     confidence -= 0.1

# if len(main_events) == 0:
#     confidence -= 0.3

# if len(locations) == 0:
#     confidence -= 0.1

# if project_name is None:
#     confidence -= 0.2

# if promoter is None:
#     confidence -= 0.1

# if len(main_events) == 0:
#     confidence -= 0.3

# if len(locations) == 0:
#     confidence -= 0.1

# df[
#     (df["relevance_confidence"] < 0.7)
#     | (df["extraction_confidence"] < 0.7)
# ]

## 3. Agente

### Build Agent

In [25]:
def build_agent(
    model,
    output_type: type[BaseModel],
    instructions: str,
    *,
    retries: int = 3,
) -> Agent:
    return Agent(
        model,
        output_type=output_type,
        instructions=instructions,
        retries=retries,
    )

In [26]:
# Para Ollama

from pydantic_ai.models.ollama import OllamaModel
from pydantic_ai.providers.ollama import OllamaProvider


def build_ollama_model(model_name: str) -> OllamaModel:
    return OllamaModel(
        model_name,
        provider=OllamaProvider(
            base_url="http://localhost:11434/v1",
        ),
    )

### Modelos disponibles

In [27]:
# MODEL_PROVIDER = "ollama"
MODEL_PROVIDER = "gemini"

In [28]:
if MODEL_PROVIDER == "gemini":
    AI_MODEL_NAME = "google:gemini-2.5-flash"
    AI_MODEL = AI_MODEL_NAME

elif MODEL_PROVIDER == "ollama":
    AI_MODEL_NAME = "qwen3:8b"
    AI_MODEL = build_ollama_model(AI_MODEL_NAME)

else:
    raise ValueError(
        f"Proveedor de modelo no soportado: {MODEL_PROVIDER}"
    )

In [29]:
agent = Agent(
    AI_MODEL,
    output_type=BOEProjectExtraction,
    instructions=INSTRUCTIONS,
    retries=3
)

## 4. Herramientas del agente

### Localización

In [30]:
municipios_ine_df = pd.read_parquet(DIM_MUNICIPALITIES_PATH)

In [31]:
# Palabras con poco valor discriminante para identificar municipios.
STOP_TOKENS = {
    "a", "de", "del", "el", "en", "la", "las", "los", "y",
    "municipio", "municipal", "termino",
}


def text_tokens(text: str | None) -> set[str]:
    """
    Convierte un texto en un conjunto de tokens normalizados,
    eliminando palabras poco informativas.
    """
    return {
        token
        for token in normalize_text(text).split()
        if token not in STOP_TOKENS
    }


def token_overlap_score(query: str | None, candidate: str | None) -> float:
    """
    Calcula la proporción de tokens de la consulta presentes
    en el candidato.

    Valor entre 0 y 1.
    """
    query_tokens = text_tokens(query)
    candidate_tokens = text_tokens(candidate)

    if not query_tokens or not candidate_tokens:
        return 0.0

    return len(query_tokens & candidate_tokens) / len(query_tokens)


def municipality_token_matches(
    municipality_name: str,
    candidate_municipality: str,
    *,
    allow_single_token: bool,
) -> bool:
    """
    Determina si un municipio candidato es compatible con la consulta.

    Reglas:
    - Coincidencia total de tokens -> match.
    - Coincidencia >= 80 % para consultas con varios tokens -> match.
    - Consultas de un solo token solo se aceptan si existen hints
      adicionales (provincia o comunidad autónoma).
    """
    query_tokens = text_tokens(municipality_name)
    candidate_tokens = text_tokens(candidate_municipality)

    if not query_tokens or not candidate_tokens:
        return False

    if query_tokens.issubset(candidate_tokens):
        return True

    if len(query_tokens) >= 2:
        return token_overlap_score(municipality_name, candidate_municipality) >= 0.8

    return allow_single_token and bool(query_tokens & candidate_tokens)


def hint_token_matches(
    hint: str | None,
    candidate: str | None,
) -> bool:
    """
    Comprueba si un hint administrativo (provincia o comunidad autónoma)
    comparte al menos un token relevante con el candidato.
    """
    if hint is None:
        return True

    hint_tokens = text_tokens(hint)
    candidate_tokens = text_tokens(candidate)

    if not hint_tokens or not candidate_tokens:
        return False

    return bool(hint_tokens & candidate_tokens)


def _row_to_location(row: pd.Series) -> MunicipalityLocation:
    """
    Convierte una fila de la dimensión INE en un objeto tipado.
    """
    return MunicipalityLocation(
        municipality=row["municipio"],
        province=row["provincia"],
        autonomous_community=row["comunidad_autonoma"],
        ine_municipality_code=row["cpro"] + row["cmun"],
        ine_province_code=row["cpro"],
        ine_autonomous_community_code=row["cauto"],
    )


def _build_lookup_result(
    municipality_name: str,
    province_hint: str | None,
    autonomous_community_hint: str | None,
    matches: pd.DataFrame,
    matched_by: str,
    reason: str,
) -> MunicipalityLookupResult:
    """
    Construye la respuesta final a partir de las coincidencias obtenidas.

    - 1 coincidencia  -> RESOLVED
    - >1 coincidencia -> AMBIGUOUS
    - 0 coincidencias -> NOT_FOUND
    """
    matches = matches.drop_duplicates(
        subset=["cauto", "cpro", "cmun"]
    )

    if len(matches) == 1:
        return MunicipalityLookupResult(
            query=municipality_name,
            province_hint=province_hint,
            autonomous_community_hint=autonomous_community_hint,
            resolution_status=MunicipalityResolutionStatus.RESOLVED,
            resolved=_row_to_location(matches.iloc[0]),
            matched_by=matched_by,
            reason=reason,
        )

    if len(matches) > 1:
        return MunicipalityLookupResult(
            query=municipality_name,
            province_hint=province_hint,
            autonomous_community_hint=autonomous_community_hint,
            resolution_status=MunicipalityResolutionStatus.AMBIGUOUS,
            candidates=[_row_to_location(row) for _, row in matches.iterrows()],
            matched_by=matched_by,
            reason=(
                "Existen varias coincidencias compatibles con los criterios "
                "proporcionados."
            ),
        )

    return MunicipalityLookupResult(
        query=municipality_name,
        province_hint=province_hint,
        autonomous_community_hint=autonomous_community_hint,
        resolution_status=MunicipalityResolutionStatus.NOT_FOUND,
        reason="No existe coincidencia en el catálogo INE.",
    )


def resolve_municipality_impl(
    municipality_name: str,
    province_hint: str | None = None,
    autonomous_community_hint: str | None = None,
) -> MunicipalityLookupResult:
    # Normalizar consulta y hints para matching.
    municipality_name_norm = normalize_text(municipality_name)
    province_hint_norm = normalize_text(province_hint) if province_hint else None
    autonomous_community_hint_norm = (
        normalize_text(autonomous_community_hint)
        if autonomous_community_hint
        else None
    )

    # Fase 1: coincidencia exacta por nombre normalizado de municipio.
    matches = municipios_ine_df.loc[
        municipios_ine_df["municipio_norm"] == municipality_name_norm
    ]

    # Fase 2: desambiguar coincidencias exactas mediante provincia.
    if not matches.empty and province_hint_norm:
        province_matches = matches.loc[
            matches["provincia"].map(
                lambda value: hint_token_matches(province_hint, value)
            )
        ]

        if not province_matches.empty:
            return _build_lookup_result(
                municipality_name=municipality_name,
                province_hint=province_hint,
                autonomous_community_hint=autonomous_community_hint,
                matches=province_matches,
                matched_by="municipality_exact_and_province_hint",
                reason=(
                    "Municipio resuelto por coincidencia exacta de municipio "
                    "y provincia compatible por tokens."
                ),
            )

    # Fase 3: desambiguar coincidencias exactas mediante comunidad autónoma.
    if not matches.empty and autonomous_community_hint_norm:
        ac_matches = matches.loc[
            matches["comunidad_autonoma"].map(
                lambda value: hint_token_matches(
                    autonomous_community_hint,
                    value,
                )
            )
        ]

        if not ac_matches.empty:
            return _build_lookup_result(
                municipality_name=municipality_name,
                province_hint=province_hint,
                autonomous_community_hint=autonomous_community_hint,
                matches=ac_matches,
                matched_by="municipality_exact_and_autonomous_community_hint",
                reason=(
                    "Municipio resuelto por coincidencia exacta de municipio "
                    "y comunidad autónoma compatible por tokens."
                ),
            )

    # Fase 4: si la coincidencia exacta ya es única, resolver.
    if not matches.empty:
        return _build_lookup_result(
            municipality_name=municipality_name,
            province_hint=province_hint,
            autonomous_community_hint=autonomous_community_hint,
            matches=matches,
            matched_by="municipality_exact",
            reason="Municipio resuelto por coincidencia exacta.",
        )

    # Permitir búsquedas de un solo token únicamente cuando existen hints.
    allow_single_token = (
        province_hint is not None
        or autonomous_community_hint is not None
    )

    # Fase 5: búsqueda flexible por tokens del municipio.
    partial_matches = municipios_ine_df.loc[
        municipios_ine_df["municipio"].map(
            lambda value: municipality_token_matches(
                municipality_name,
                value,
                allow_single_token=allow_single_token,
            )
        )
    ]

    # Fase 6: filtrar coincidencias parciales mediante provincia.
    if not partial_matches.empty and province_hint is not None:
        province_partial_matches = partial_matches.loc[
            partial_matches["provincia"].map(
                lambda value: hint_token_matches(province_hint, value)
            )
        ]

        if not province_partial_matches.empty:
            return _build_lookup_result(
                municipality_name=municipality_name,
                province_hint=province_hint,
                autonomous_community_hint=autonomous_community_hint,
                matches=province_partial_matches,
                matched_by="municipality_token_and_province_hint",
                reason=(
                    "Municipio resuelto por coincidencia de tokens del municipio "
                    "y provincia compatible por tokens."
                ),
            )

    # Fase 7: filtrar coincidencias parciales mediante comunidad autónoma.
    if not partial_matches.empty and autonomous_community_hint is not None:
        ac_partial_matches = partial_matches.loc[
            partial_matches["comunidad_autonoma"].map(
                lambda value: hint_token_matches(
                    autonomous_community_hint,
                    value,
                )
            )
        ]

        if not ac_partial_matches.empty:
            return _build_lookup_result(
                municipality_name=municipality_name,
                province_hint=province_hint,
                autonomous_community_hint=autonomous_community_hint,
                matches=ac_partial_matches,
                matched_by="municipality_token_and_autonomous_community_hint",
                reason=(
                    "Municipio resuelto por coincidencia de tokens del municipio "
                    "y comunidad autónoma compatible por tokens."
                ),
            )

    # Fase 8: devolver coincidencias parciales restantes.
    if not partial_matches.empty:
        return _build_lookup_result(
            municipality_name=municipality_name,
            province_hint=province_hint,
            autonomous_community_hint=autonomous_community_hint,
            matches=partial_matches,
            matched_by="municipality_token",
            reason=(
                "Existen coincidencias por tokens del municipio, pero no hay "
                "hints suficientes para garantizar una resolución única."
            ),
        )

    # Fase 9: sin coincidencias exactas ni parciales.
    return MunicipalityLookupResult(
        query=municipality_name,
        province_hint=province_hint,
        autonomous_community_hint=autonomous_community_hint,
        resolution_status=MunicipalityResolutionStatus.NOT_FOUND,
        reason="No existe coincidencia exacta ni por tokens en el catálogo INE.",
    )

In [32]:
@agent.tool_plain
def resolve_municipality(
    municipality_name: str,
    province_hint: str | None = None,
    autonomous_community_hint: str | None = None,
) -> MunicipalityLookupResult:
    return resolve_municipality_impl(
        municipality_name=municipality_name,
        province_hint=province_hint,
        autonomous_community_hint=autonomous_community_hint,
    )

# resolve_municipality("Amurrio")
# resolve_municipality("Agurain/Salvatierra")
# resolve_municipality("Palmas")
# resolve_municipality("Palmas", province_hint="Las Palmas")
# resolve_municipality("Gran Canaria", province_hint="Las Palmas")
# resolve_municipality("Palmas", autonomous_community_hint="Canarias")

## 5. Extracción con IA

In [35]:
df = pd.read_parquet(BOE_CANDIDATES_DOCS_TEXT_PATH)

df_test = df.loc[df["xml_status"] == "ok"]

df_test.head(2)

,identificador,doc_file_stem,url_html,url_xml,fecha_publicacion,titulo,epigrafe_nombre,departamento_nombre,seccion_nombre,xml_path,texto_limpio,texto_len,xml_status,parse_error,parsed_at
0,BOE-B-2021-32554,20210707_BOE-B-2021-32554,https://www.boe.es/diario_boe/txt.php?id=BOE-B...,https://www.boe.es/diario_boe/xml.php?id=BOE-B...,2021-07-07,Resolución de la Dirección General de Planific...,None,"MINISTERIO DE TRANSPORTES, MOVILIDAD Y AGENDA ...",V. Anuncios. - B. Otros anuncios oficiales,/home/bgonzale/CiDaeN/15_TrabajoFinMaster/tfm-...,"BOE-B-2021-32554 Ministerio de Transportes, Mo...",10752,ok,None,2026-06-18T11:07:32.082516+00:00
1,BOE-B-2021-32555,20210707_BOE-B-2021-32555,https://www.boe.es/diario_boe/txt.php?id=BOE-B...,https://www.boe.es/diario_boe/xml.php?id=BOE-B...,2021-07-07,Resolución de la Dirección General de Planific...,None,"MINISTERIO DE TRANSPORTES, MOVILIDAD Y AGENDA ...",V. Anuncios. - B. Otros anuncios oficiales,/home/bgonzale/CiDaeN/15_TrabajoFinMaster/tfm-...,"BOE-B-2021-32555 Ministerio de Transportes, Mo...",10752,ok,None,2026-06-18T11:07:32.082516+00:00


In [ ]:
# for _, row in df_test.sample(10, random_state=42).iterrows():
#     print("=" * 80)
#     print(row["identificador"])
#     print(row["titulo"])
#     print(row["texto_limpio"][:6000])

In [36]:
# df_test2 = df_test.loc[df_test["identificador"] == "BOE-A-2023-10306"]
df_test2 = df_test.loc[df_test["identificador"] == "BOE-A-2023-2598"]

In [37]:
df_test2.values

array([['BOE-A-2023-2598', '20230131_BOE-A-2023-2598',
        'https://www.boe.es/diario_boe/txt.php?id=BOE-A-2023-2598',
        'https://www.boe.es/diario_boe/xml.php?id=BOE-A-2023-2598',
        '2023-01-31',
        'Resolución de 23 de enero de 2023, de la Dirección General de Calidad y Evaluación Ambiental, por la que se formula la declaración de impacto ambiental del proyecto "Parque eólico Badulaque de 90 MW, y su infraestructura de evacuación, en los Concellos de Valdoviño, Cerdido, Cerdeira, Moeche, As Somozas y As Pontes de García Rodríguez (A Coruña)".',
        'Impacto ambiental',
        'MINISTERIO PARA LA TRANSICIÓN ECOLÓGICA Y EL RETO DEMOGRÁFICO',
        'III. Otras disposiciones',
        '/home/bgonzale/CiDaeN/15_TrabajoFinMaster/tfm-renewables-permitting-tracker/data/bronze/boe_docs_xml/20230131_BOE-A-2023-2598.xml',
        'BOE-A-2023-2598 Estatal Ministerio para la Transición Ecológica y el Reto Demográfico Resolución 20230123 Resolución de 23 de enero de 202

In [40]:
for _, row in df_test2.sample(1, random_state=42).iterrows():
    prompt = f"""
        Identificador BOE: {row["identificador"]}
        Fecha publicación: {row["fecha_publicacion"]}
        Título: {row["titulo"]}

        Texto:
        {row["texto_limpio"][:3000]}  # 12000
        """
    result = await agent.run(prompt)

### Ver el resultado de la extracción (extraction_json)

In [ ]:
resultoutput = result.output

my_result = result.output.model_dump_json()

In [ ]:
my_result_dict = json.loads(my_result)

print(
    json.dumps(
        my_result_dict,
        indent=2,
        ensure_ascii=False,
    )
)

{
  "identificador_boe": "BOE-A-2023-2598",
  "fecha_publicacion": "2023-01-31",
  "relevancia_energetica": "relevante",
  "es_relevante_para_proyecto": true,
  "relevance_reason": "El documento formula la declaración de impacto ambiental de un proyecto de parque eólico.",
  "lifecycle_events": [
    {
      "event_type": "new_project",
      "administrative_actions": [
        {
          "procedure_stage": "declaracion_impacto_ambiental",
          "procedure_decision": "formulado",
          "evidence": "Resolución de 23 de enero de 2023, de la Dirección General de Calidad y Evaluación Ambiental, por la que se formula la declaración de impacto ambiental del proyecto \"Parque eólico Badulaque de 90 MW...\""
        }
      ],
      "assets": [
        {
          "local_asset_id": "asset_1",
          "name": "Parque Eólico Badulaque",
          "aliases": [
            "PE Badulaque"
          ],
          "role_in_event": "new_asset",
          "status_in_document": "under_permitti

## 6. Guardar y leer lo extraído

### Funciones

#### Registro de extracción

In [44]:
def build_ai_extraction_record(
    row: pd.Series,
    extraction: BOEProjectExtraction,
    model_name: str,
) -> dict:
    """
    Construye un registro tabular a partir de una extracción IA validada.

    La función transforma el objeto Pydantic `BOEProjectExtraction` en una
    fila apta para almacenarse en la capa silver `boe_ai_extractions`.

    Parameters
    ----------
    row : pd.Series
        Fila original del documento BOE procedente de `boe_candidates_docs_text`.
        Debe contener, al menos, la columna `titulo`.
    extraction : BOEProjectExtraction
        Resultado estructurado devuelto por el agente y validado por Pydantic.
    model_name : str
        Nombre del modelo utilizado para generar la extracción.

    Returns
    -------
    dict
        Registro con metadatos de extracción y JSON completo serializado.
    """
    return {
        "identificador_boe": extraction.identificador_boe,
        "fecha_publicacion": extraction.fecha_publicacion,
        "titulo": row["titulo"],
        "extraction_json": extraction.model_dump_json(),
        "extracted_at": datetime.now(timezone.utc).isoformat(),
        "model_name": model_name,
        "extraction_status": "ok",
        "parse_error": None,
    }

#### Upsert incremental

In [45]:
def upsert_ai_extractions(
    new_ai_extractions: pd.DataFrame,
    output_path: Path,
) -> pd.DataFrame:
    """
    Inserta o actualiza extracciones IA en un Parquet acumulado.

    Si el fichero ya existe, concatena las nuevas extracciones con las
    existentes y conserva la versión más reciente de cada `identificador_boe`.
    Si no existe, crea el fichero desde cero.

    Esta función debe usarse solo para la tabla fuente
    `boe_ai_extractions.parquet`. Las tablas derivadas deben regenerarse
    a partir de esta tabla, no actualizarse incrementalmente.

    Parameters
    ----------
    new_ai_extractions : pd.DataFrame
        Nuevas extracciones IA a incorporar. Debe contener la columna
        `identificador_boe`.
    output_path : Path
        Ruta del Parquet acumulado de salida.

    Returns
    -------
    pd.DataFrame
        DataFrame completo actualizado tras aplicar el upsert.
    """
    if output_path.exists():
        existing_ai_extractions = pd.read_parquet(output_path)

        ai_extractions = pd.concat(
            [existing_ai_extractions, new_ai_extractions],
            ignore_index=True,
        )

        ai_extractions = ai_extractions.drop_duplicates(
            subset=["identificador_boe"],
            keep="last",
        )
    else:
        ai_extractions = new_ai_extractions.copy()

    save_parquet(ai_extractions, output_path)

    return ai_extractions

### Prueba

In [46]:
extraction = result.output

record = build_ai_extraction_record(
    row=row,
    extraction=extraction,
    model_name=AI_MODEL_NAME,
)

new_ai_extractions = pd.DataFrame([record])

ai_extractions = upsert_ai_extractions(
    new_ai_extractions,
    BOE_AI_EXTRACTIONS_PATH,
)

In [47]:
pd.read_parquet(BOE_AI_EXTRACTIONS_PATH)

,identificador_boe,fecha_publicacion,titulo,extraction_json,extracted_at,model_name,extraction_status,parse_error
0,BOE-A-2023-10306,2023-04-28,"Resolución de 17 de abril de 2023, de la Direc...","{""identificador_boe"":""BOE-A-2023-10306"",""fecha...",2026-06-19T14:45:04.148710+00:00,google:gemini-2.5-flash,ok,None
1,BOE-A-2023-2598,2023-01-31,"Resolución de 23 de enero de 2023, de la Direc...","{""identificador_boe"":""BOE-A-2023-2598"",""fecha_...",2026-06-19T15:24:33.589510+00:00,google:gemini-2.5-flash,ok,None


## 7. Flattening data in field extraction_json

### Funciones

#### flatten_lifecycle_events

In [48]:
def flatten_lifecycle_events(ai_extractions: pd.DataFrame) -> pd.DataFrame:
    records = []

    for _, row in ai_extractions.iterrows():
        extraction = BOEProjectExtraction.model_validate_json(
            row["extraction_json"]
        )

        for event_idx, event in enumerate(
            extraction.lifecycle_events,
            start=1,
        ):
            event_id = f"{extraction.identificador_boe}_event_{event_idx}"

            records.append(
                {
                    "event_id": event_id,
                    "identificador_boe": extraction.identificador_boe,
                    "fecha_publicacion": extraction.fecha_publicacion,
                    "event_index": event_idx,
                    "event_type": event.event_type.value,
                    "event_summary": event.event_summary,
                    "evidence": event.evidence,
                }
            )

    return pd.DataFrame(records)

#### flatten_administrative_actions

In [49]:
def flatten_administrative_actions(ai_extractions: pd.DataFrame) -> pd.DataFrame:
    records = []

    for _, row in ai_extractions.iterrows():
        extraction = BOEProjectExtraction.model_validate_json(
            row["extraction_json"]
        )

        for event_idx, event in enumerate(
            extraction.lifecycle_events,
            start=1,
        ):
            event_id = f"{extraction.identificador_boe}_event_{event_idx}"

            for action_idx, action in enumerate(
                event.administrative_actions,
                start=1,
            ):
                action_id = (
                    f"{extraction.identificador_boe}"
                    f"_event_{event_idx}"
                    f"_action_{action_idx}"
                )

                records.append(
                    {
                        "action_id": action_id,
                        "event_id": event_id,
                        "identificador_boe": extraction.identificador_boe,
                        "fecha_publicacion": extraction.fecha_publicacion,
                        "action_index": action_idx,
                        "procedure_stage": action.procedure_stage.value,
                        "procedure_decision": action.procedure_decision.value,
                        "evidence": action.evidence,
                    }
                )

    return pd.DataFrame(records)

#### flatten_asset_mentions

In [ ]:
def flatten_asset_mentions(ai_extractions: pd.DataFrame) -> pd.DataFrame:
    records = []

    for _, row in ai_extractions.iterrows():
        extraction = BOEProjectExtraction.model_validate_json(row["extraction_json"])

        for event_idx, event in enumerate(extraction.lifecycle_events, start=1):
            event_id = f"{extraction.identificador_boe}_event_{event_idx}"

            for asset in event.assets:
                asset_mention_id = f"{event_id}_{asset.local_asset_id}"

                records.append(
                    {
                        "asset_mention_id": asset_mention_id,
                        "event_id": event_id,
                        "identificador_boe": extraction.identificador_boe,
                        "fecha_publicacion": extraction.fecha_publicacion,
                        "local_asset_id": asset.local_asset_id,
                        "asset_name": asset.name,
                        "asset_name_norm": normalize_text(asset.name) if asset.name else None,
                        "role_in_event": asset.role_in_event.value,
                        "status_in_document": asset.status_in_document.value,
                        "evidence": asset.evidence,
                    }
                )

    return pd.DataFrame(records)

#### flatten_asset_technologies

In [51]:
def flatten_asset_technologies(ai_extractions: pd.DataFrame) -> pd.DataFrame:
    records = []

    for _, row in ai_extractions.iterrows():
        extraction = BOEProjectExtraction.model_validate_json(row["extraction_json"])

        for event_idx, event in enumerate(extraction.lifecycle_events, start=1):
            event_id = f"{extraction.identificador_boe}_event_{event_idx}"

            for asset in event.assets:
                asset_mention_id = f"{event_id}_{asset.local_asset_id}"

                for tech_idx, tech in enumerate(asset.technologies, start=1):
                    records.append(
                        {
                            "asset_technology_id": f"{asset_mention_id}_tech_{tech_idx}",
                            "asset_mention_id": asset_mention_id,
                            "event_id": event_id,
                            "identificador_boe": extraction.identificador_boe,
                            "technology_type": tech.technology_type.value,
                            "installed_power_mw": tech.installed_power_mw,
                            "peak_power_mwp": tech.peak_power_mwp,
                            "description": tech.description,
                            "power_normalization_note": tech.power_normalization_note,
                        }
                    )

    return pd.DataFrame(records)

#### flatten_asset_participants

In [52]:
def flatten_asset_participants(ai_extractions: pd.DataFrame) -> pd.DataFrame:
    records = []

    for _, row in ai_extractions.iterrows():
        extraction = BOEProjectExtraction.model_validate_json(row["extraction_json"])

        for event_idx, event in enumerate(extraction.lifecycle_events, start=1):
            event_id = f"{extraction.identificador_boe}_event_{event_idx}"

            for asset in event.assets:
                asset_mention_id = f"{event_id}_{asset.local_asset_id}"

                for participant_idx, participant in enumerate(asset.participants, start=1):
                    records.append(
                        {
                            "asset_participant_id": f"{asset_mention_id}_participant_{participant_idx}",
                            "asset_mention_id": asset_mention_id,
                            "event_id": event_id,
                            "identificador_boe": extraction.identificador_boe,
                            "participant_name": participant.name,
                            "participant_name_norm": normalize_text(participant.name) if participant.name else None,
                            "participant_role": participant.role.value,
                            "evidence": participant.evidence,
                        }
                    )

    return pd.DataFrame(records)

#### flatten_asset_locations

In [53]:
def flatten_asset_locations(ai_extractions: pd.DataFrame) -> pd.DataFrame:
    records = []

    for _, row in ai_extractions.iterrows():
        extraction = BOEProjectExtraction.model_validate_json(row["extraction_json"])

        for event_idx, event in enumerate(extraction.lifecycle_events, start=1):
            event_id = f"{extraction.identificador_boe}_event_{event_idx}"

            for asset in event.assets:
                asset_mention_id = f"{event_id}_{asset.local_asset_id}"

                for location_idx, loc in enumerate(asset.locations, start=1):
                    records.append(
                        {
                            "asset_location_id": f"{asset_mention_id}_location_{location_idx}",
                            "asset_mention_id": asset_mention_id,
                            "event_id": event_id,
                            "identificador_boe": extraction.identificador_boe,
                            "ine_municipality_code": loc.ine_municipality_code,
                            "municipality": loc.municipality,
                            "municipality_norm": normalize_text(loc.municipality) if loc.municipality else None,
                            "ine_province_code": loc.ine_province_code,
                            "province": loc.province,
                            "province_norm": normalize_text(loc.province) if loc.province else None,
                            "ine_autonomous_community_code": loc.ine_autonomous_community_code,
                            "autonomous_community": loc.autonomous_community,
                        }
                    )

    return pd.DataFrame(records)

#### flatten_asset_aliases

In [54]:
def flatten_asset_aliases(ai_extractions: pd.DataFrame) -> pd.DataFrame:
    records = []

    for _, row in ai_extractions.iterrows():
        extraction = BOEProjectExtraction.model_validate_json(row["extraction_json"])

        for event_idx, event in enumerate(extraction.lifecycle_events, start=1):
            event_id = f"{extraction.identificador_boe}_event_{event_idx}"

            for asset in event.assets:
                asset_mention_id = f"{event_id}_{asset.local_asset_id}"

                for alias_idx, alias in enumerate(asset.aliases, start=1):
                    records.append(
                        {
                            "asset_alias_id": f"{asset_mention_id}_alias_{alias_idx}",
                            "asset_mention_id": asset_mention_id,
                            "event_id": event_id,
                            "identificador_boe": extraction.identificador_boe,
                            "alias": alias,
                            "alias_norm": normalize_text(alias) if alias else None,
                        }
                    )

    return pd.DataFrame(records)

#### flatten_asset_relation_mentions

In [55]:
def flatten_asset_relation_mentions(ai_extractions: pd.DataFrame) -> pd.DataFrame:
    records = []

    for _, row in ai_extractions.iterrows():
        extraction = BOEProjectExtraction.model_validate_json(row["extraction_json"])

        for event_idx, event in enumerate(extraction.lifecycle_events, start=1):
            event_id = f"{extraction.identificador_boe}_event_{event_idx}"

            for relation_idx, relation in enumerate(event.asset_relations, start=1):
                records.append(
                    {
                        "relation_mention_id": (
                            f"{extraction.identificador_boe}"
                            f"_event_{event_idx}"
                            f"_relation_{relation_idx}"
                        ),
                        "event_id": event_id,
                        "identificador_boe": extraction.identificador_boe,
                        "fecha_publicacion": extraction.fecha_publicacion,
                        "source_asset_mention_id": (
                            f"{extraction.identificador_boe}"
                            f"_event_{event_idx}_{relation.source_asset_id}"
                        ),
                        "target_asset_mention_id": (
                            f"{extraction.identificador_boe}"
                            f"_event_{event_idx}_{relation.target_asset_id}"
                        ),
                        "source_local_asset_id": relation.source_asset_id,
                        "target_local_asset_id": relation.target_asset_id,
                        "relation_type": relation.relation_type.value,
                        "evidence": relation.evidence,
                    }
                )

    return pd.DataFrame(records)

### Prueba

In [ ]:
lifecycle_events = flatten_lifecycle_events(ai_extractions)
administrative_actions = flatten_administrative_actions(ai_extractions)
asset_mentions = flatten_asset_mentions(ai_extractions)
asset_technologies = flatten_asset_technologies(ai_extractions)
asset_participants = flatten_asset_participants(ai_extractions)
asset_locations = flatten_asset_locations(ai_extractions)
asset_aliases = flatten_asset_aliases(ai_extractions)
asset_relation_mentions = flatten_asset_relation_mentions(ai_extractions)

In [ ]:
save_parquet(lifecycle_events, LIFECYCLE_EVENTS_PATH)
save_parquet(administrative_actions, ADMINISTRATIVE_ACTIONS_PATH)
save_parquet(asset_mentions, ASSET_MENTIONS_PATH)
save_parquet(asset_technologies, ASSET_TECHNOLOGIES_PATH)
save_parquet(asset_participants, ASSET_PARTICIPANTS_PATH)
save_parquet(asset_locations, ASSET_LOCATIONS_PATH)
save_parquet(asset_aliases, ASSET_ALIASES_PATH)
save_parquet(asset_relation_mentions, ASSET_RELATION_MENTIONS_PATH)

In [60]:
display(lifecycle_events)
display(administrative_actions)
display(asset_mentions)
display(asset_technologies)
display(asset_participants)
display(asset_locations)
display(asset_aliases)
display(asset_relation_mentions)

,event_id,identificador_boe,fecha_publicacion,event_index,event_type,event_summary,evidence
0,BOE-A-2023-10306_event_1,BOE-A-2023-10306,2023-04-28,1,new_project,Se otorga autorización administrativa previa p...,"Resolución de 17 de abril de 2023, de la Direc..."
1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,2023-01-31,1,new_project,Formulación de la declaración de impacto ambie...,"Resolución de 23 de enero de 2023, de la Direc..."


,action_id,event_id,identificador_boe,fecha_publicacion,action_index,procedure_stage,procedure_decision,evidence
0,BOE-A-2023-10306_event_1_action_1,BOE-A-2023-10306_event_1,BOE-A-2023-10306,2023-04-28,1,autorizacion_administrativa_previa,favorable,por la que se otorga a Enel Green Power España...
1,BOE-A-2023-2598_event_1_action_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,2023-01-31,1,declaracion_impacto_ambiental,formulado,"Resolución de 23 de enero de 2023, de la Direc..."


,asset_mention_id,event_id,identificador_boe,fecha_publicacion,local_asset_id,asset_name,asset_name_norm,role_in_event,status_in_document,evidence
0,BOE-A-2023-10306_event_1_asset_1,BOE-A-2023-10306_event_1,BOE-A-2023-10306,2023-04-28,asset_1,Parque eólico Badulaque,parque eolico badulaque,new_asset,under_permitting,parque eólico Badulaque de 90 MW
1,BOE-A-2023-2598_event_1_asset_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,2023-01-31,asset_1,Parque Eólico Badulaque,parque eolico badulaque,new_asset,under_permitting,El proyecto de Badulaque pertenece al conjunto...


,asset_technology_id,asset_mention_id,event_id,identificador_boe,technology_type,installed_power_mw,peak_power_mwp,description,power_normalization_note
0,BOE-A-2023-10306_event_1_asset_1_tech_1,BOE-A-2023-10306_event_1_asset_1,BOE-A-2023-10306_event_1,BOE-A-2023-10306,eolica,90.0,None,parque eólico,None
1,BOE-A-2023-2598_event_1_asset_1_tech_1,BOE-A-2023-2598_event_1_asset_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,eolica,90.0,None,20 aerogeneradores titulares y 8 aerogenerador...,The document states a total power of 90 MW for...


,asset_participant_id,asset_mention_id,event_id,identificador_boe,participant_name,participant_name_norm,participant_role,evidence
0,BOE-A-2023-10306_event_1_asset_1_participant_1,BOE-A-2023-10306_event_1_asset_1,BOE-A-2023-10306_event_1,BOE-A-2023-10306,"Enel Green Power España, SL",enel green power espana sl,promoter,"se otorga a Enel Green Power España, SL, autor..."
1,BOE-A-2023-2598_event_1_asset_1_participant_1,BOE-A-2023-2598_event_1_asset_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,Enel Green Power S.L.,enel green power s l,promoter,promovido por Enel Green Power S.L.


,asset_location_id,asset_mention_id,event_id,identificador_boe,ine_municipality_code,municipality,municipality_norm,ine_province_code,province,province_norm,ine_autonomous_community_code,autonomous_community
0,BOE-A-2023-10306_event_1_asset_1_location_1,BOE-A-2023-10306_event_1_asset_1,BOE-A-2023-10306_event_1,BOE-A-2023-10306,15087,Valdoviño,valdovino,15,"Coruña, A",coruna a,12,Galicia
1,BOE-A-2023-10306_event_1_asset_1_location_2,BOE-A-2023-10306_event_1_asset_1,BOE-A-2023-10306_event_1,BOE-A-2023-10306,15025,Cerdido,cerdido,15,"Coruña, A",coruna a,12,Galicia
2,BOE-A-2023-10306_event_1_asset_1_location_3,BOE-A-2023-10306_event_1_asset_1,BOE-A-2023-10306_event_1,BOE-A-2023-10306,15022,Cedeira,cedeira,15,"Coruña, A",coruna a,12,Galicia
3,BOE-A-2023-10306_event_1_asset_1_location_4,BOE-A-2023-10306_event_1_asset_1,BOE-A-2023-10306_event_1,BOE-A-2023-10306,15049,Moeche,moeche,15,"Coruña, A",coruna a,12,Galicia
4,BOE-A-2023-10306_event_1_asset_1_location_5,BOE-A-2023-10306_event_1_asset_1,BOE-A-2023-10306_event_1,BOE-A-2023-10306,15081,"Somozas, As",somozas as,15,"Coruña, A",coruna a,12,Galicia
5,BOE-A-2023-10306_event_1_asset_1_location_6,BOE-A-2023-10306_event_1_asset_1,BOE-A-2023-10306_event_1,BOE-A-2023-10306,15070,"Pontes de García Rodríguez, As",pontes de garcia rodriguez as,15,"Coruña, A",coruna a,12,Galicia
6,BOE-A-2023-2598_event_1_asset_1_location_1,BOE-A-2023-2598_event_1_asset_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,15087,Valdoviño,valdovino,15,"Coruña, A",coruna a,12,Galicia
7,BOE-A-2023-2598_event_1_asset_1_location_2,BOE-A-2023-2598_event_1_asset_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,15025,Cerdido,cerdido,15,"Coruña, A",coruna a,12,Galicia
8,BOE-A-2023-2598_event_1_asset_1_location_3,BOE-A-2023-2598_event_1_asset_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,15022,Cedeira,cedeira,15,"Coruña, A",coruna a,12,Galicia
9,BOE-A-2023-2598_event_1_asset_1_location_4,BOE-A-2023-2598_event_1_asset_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,15049,Moeche,moeche,15,"Coruña, A",coruna a,12,Galicia


,asset_alias_id,asset_mention_id,event_id,identificador_boe,alias,alias_norm
0,BOE-A-2023-2598_event_1_asset_1_alias_1,BOE-A-2023-2598_event_1_asset_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,PE Badulaque,pe badulaque


""
